# 01 — Data Exploration

Explore the video dataset and understand the distribution of video quality metrics
before training. This mirrors `01_data_exploration.ipynb` in the text RLHF repo
but for the video modality.

**What we look at:**
1. Dataset overview — how many videos, what domain, caption length distribution
2. Frame quality — brightness, sharpness, contrast distribution
3. Automated quality metrics on a sample — motion smoothness, temporal consistency, CLIP score
4. Visualize the metric distributions to understand the signal-to-noise ratio

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from data.video_dataset import VideoDataset, load_video_frames
from data.video_metrics import compute_video_metrics

plt.style.use('seaborn-v0_8-whitegrid')
print('Imports OK')

In [ ]:
# Load dataset
dataset = VideoDataset(
    video_dir='../data/videos',
    captions_path='../data/captions.json',
    num_frames=16,
    height=480,
    width=720,
    max_samples=200,  # sample for exploration
)

print(f'Dataset size: {len(dataset)} videos')

# Caption length distribution
captions = [s[1] for s in dataset.samples]
lengths = [len(c.split()) for c in captions]

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lengths, bins=20, color='#4C72B0', alpha=0.8, edgecolor='white')
ax.set_xlabel('Caption length (words)')
ax.set_ylabel('Count')
ax.set_title(f'Caption length distribution (mean={np.mean(lengths):.1f} words)')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample frames from a few videos
n_videos = 4
n_frames = 4

fig, axes = plt.subplots(n_videos, n_frames, figsize=(16, 10))

for i in range(n_videos):
    sample = dataset[i * (len(dataset) // n_videos)]
    frames = sample['pixel_values']  # (T, C, H, W) in [-1, 1]
    caption = sample['caption']
    
    indices = np.linspace(0, len(frames) - 1, n_frames, dtype=int)
    for j, fidx in enumerate(indices):
        frame = ((frames[fidx].permute(1, 2, 0) + 1.0) * 127.5).clamp(0, 255).byte().numpy()
        axes[i, j].imshow(frame)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(caption[:40] + '...', fontsize=8, ha='left', x=0)

plt.suptitle('Sample video frames', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compute quality metrics on a 30-video sample
import torch

SAMPLE_SIZE = 30
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

metric_rows = []
for i in range(SAMPLE_SIZE):
    sample = dataset[i * (len(dataset) // SAMPLE_SIZE)]
    frames = sample['pixel_values']
    caption = sample['caption']
    
    metrics = compute_video_metrics(frames, caption, device=DEVICE)
    metric_rows.append(metrics.to_dict())
    print(f'  [{i+1}/{SAMPLE_SIZE}] motion={metrics.motion_smoothness:.3f} '
          f'temporal={metrics.temporal_consistency:.3f} clip={metrics.prompt_adherence:.3f}')

import pandas as pd
df = pd.DataFrame(metric_rows)
print('\nMetric statistics:')
print(df.describe().round(4))

In [ ]:
# Plot metric distributions
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
metric_names = ['motion_smoothness', 'temporal_consistency', 'prompt_adherence', 'composite']
titles = ['Motion Smoothness', 'Temporal Consistency', 'CLIP Score', 'Composite Reward']

for ax, col, color, title in zip(axes, metric_names, colors, titles):
    ax.hist(df[col], bins=15, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(df[col].mean(), color='black', linestyle='--', label=f'mean={df[col].mean():.3f}')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.set_xlim(0, 1)
    ax.legend(fontsize=8)

plt.suptitle('Quality metric distributions (base CogVideoX-2B, no fine-tuning)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey observation: CLIP score variance shows room for improvement via RLHF.')
print('Motion smoothness is already high (diffusion models are temporally coherent).')
print('Temporal consistency is the noisiest signal — most sensitive to domain mismatch.')